# 08 — Activity and centroid-separation metrics

Beyond R-squared, `FittingMetricsCalculator` produces an activity map (count of active Gaussians per pixel, amplitude above threshold), per-component amplitude / mu / sigma maps, and adjacent centroid-separation maps. This notebook builds a parameter field where the active-component count and centroid separations are known by construction and checks the metric reproduces them.

Modules exercised:

- `pipelines.param_pipeline.metrics.FittingMetricsCalculator` (real, imported and run)
- `tools.logger.Logger` (real, imported)

Synthetic tomogram, fixed seed, CPU only.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


In [ ]:
import tempfile
from pathlib import Path

from tools.gaussians import GaussianMixture
from pipelines.param_pipeline.metrics import FittingMetricsCalculator
from tools.logger import Logger

logger = Logger(log_dir='', name='nb08_metrics')
print('imported')

## Parameter field with a known activity pattern

We split the range axis into three bands so that the left third has one active Gaussian, the middle third has two, and the right third has three. Inactive components are given zero amplitude so they fall below the activity threshold (`1e-3`). The second-vs-first centroid separation in the two/three-active bands is fixed at a known value.

In [ ]:
K            = 3
Az, R        = 12, 30
H            = 80
height_range = (0.0, 45.0)
height_axis  = np.linspace(*height_range, H).astype(np.float32)
third        = R // 3

params = np.zeros((3 * K, Az, R), dtype=np.float32)

# component 0 active everywhere
params[0, :, :] = 1.0
params[1, :, :] = 10.0
params[2, :, :] = 2.5

# component 1 active in middle and right thirds, centroid 18 m above comp 0
params[3, :, third:] = 0.8
params[4, :, third:] = 28.0
params[5, :, third:] = 3.0

# component 2 active in right third only
params[6, :, 2 * third:] = 0.6
params[7, :, 2 * third:] = 40.0
params[8, :, 2 * third:] = 2.0

tomogram = np.zeros((H, Az, R), dtype=np.float32)
for j, h in enumerate(height_axis):
    tomogram[j] = GaussianMixture.evaluate_slice(params, float(h), K)
tomogram += rng.normal(0.0, 0.005, tomogram.shape).astype(np.float32)

tmp_dir  = Path(tempfile.mkdtemp())
tom_path = tmp_dir / 'banded_tomogram.npy'
np.save(tom_path, tomogram)
metadata = {'height_range': list(height_range)}
print('built banded parameter field, R thirds at', third, 2 * third)

In [ ]:
calc    = FittingMetricsCalculator(n_gaussians=K, logger=logger)
metrics = calc.run(params, metadata, tom_path)

activity = metrics['activity_map']
sep01    = metrics['mu_sep_0_1']
print('activity per range band (column means):')
print('  left   third:', float(np.nanmean(activity[:, :third])))
print('  middle third:', float(np.nanmean(activity[:, third:2 * third])))
print('  right  third:', float(np.nanmean(activity[:, 2 * third:])))
print('mu separation (comp0->comp1) where both active:', float(np.nanmean(sep01)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

im0 = axes[0].imshow(activity, cmap='viridis', aspect='auto', vmin=0, vmax=K)
axes[0].set_title('Active Gaussian count per pixel')
axes[0].set_xlabel('range [px]'); axes[0].set_ylabel('azimuth [px]')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04).set_label('active K')

im1 = axes[1].imshow(sep01, cmap='magma', aspect='auto')
axes[1].set_title('|mu_2 - mu_1| where both active')
axes[1].set_xlabel('range [px]'); axes[1].set_ylabel('azimuth [px]')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04).set_label('separation [m]')
fig.tight_layout()
plt.show()

## Expected outcome

The activity map should show three clean vertical bands with values 1, 2, and 3 from left to right. The separation map should be undefined (masked) in the left band where the second component is inactive and equal to roughly 18 m (28 minus 10) in the middle and right bands, confirming both the activity counting and the centroid-separation computation.